In [ ]:
import pandas as pd
import json
import os
import psycopg2

In [ ]:
path = "/Applications/python/nutristeppe/data/settings"
rows = []

for filename in os.listdir(path):
    if filename.endswith(".json"):
        bls_code = filename.replace(".json", "")
        
        with open(os.path.join(path, filename), "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if isinstance(data, list):
            data = data[0]
        
        per100 = data.get("nutrition_per_100g", {})
        
        ingredients = [ing["name"] for ing in data.get("ingredients", [])]
        
        rows.append({
            "bls_code":         bls_code,
            "protein":          per100.get("protein", {}).get("value", 0.0),
            "fat":              per100.get("fat", {}).get("value", 0.0),
            "carbohydrate":     per100.get("carbohydrate", {}).get("value", 0.0),
            "fiber":            per100.get("fiber", {}).get("value", 0.0),
            "sugar_mg":            per100.get("Сахар (общий)", {}).get("value", 0.0),
            "salt_total_mg":       per100.get("Поваренная соль всего", {}).get("value", 0.0),
            "saturated_fat_mg":    per100.get("Насыщенные жирные кислоты", {}).get("value", 0.0),
            "kilocalories":     data.get("calories_kcal_total_for_yield", 0.0),
            "ingredients":      ingredients,
            "techcard":         data.get("technology", ""),
        })

json_df = pd.DataFrame(rows)
json_df

,bls_code,protein,fat,carbohydrate,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,kilocalories,ingredients,techcard
0,Y363323,10.620,9.562,2.909,1.112,2.305,0.386,2.766,140.20,"[свиная рулька, морковь, лук репчатый, перец с...",Свиную рульку нарезают порционными кусками и о...
1,U501262,20.600,5.444,1.287,0.225,0.668,0.408,1.898,136.60,"[свинина, соль, перец черный, чеснок, паприка]","Свинину нарезать на порционные куски, натереть..."
2,Y221232,18.030,4.499,0.287,0.019,0.087,0.412,1.323,113.80,"[телятина, масло растительное, соль, перец чер...","Отбивную нарезать порционно, слегка отбить, по..."
3,U511262,21.490,3.952,0.002,0.001,0.002,0.408,0.883,121.60,"[Филе свинины нежирное, соль, перец черный, ма...","Филе свинины промыть, обсушить. Натрите солью ..."
4,16-041,11.710,1.840,0.125,0.042,0.122,0.296,0.419,63.88,"[филе камбалы, соль, перец черный, лимон, зелень]","Филе камбалы обсушить, равномерно распределить..."
...,...,...,...,...,...,...,...,...,...,...,...
2613,15-749,1.315,3.493,8.539,0.716,1.960,0.302,0.322,70.85,"[кабачок, картофель, лук репчатый, масло расти...","1. Лук репчатый, чеснок очистить и нарезать. К..."
2614,58160510,5.703,2.483,50.100,3.030,2.497,0.311,0.227,245.60,"[рис, морковь, зеленый горошек, масло растител...",Рис отваривают до готовности. Морковь нарезают...
2615,58164800,7.258,3.977,66.780,1.982,1.032,0.329,1.771,331.90,"[рис коричневый, Сливки 20% жирности, соль]","Рис промыть, затем отварить в подсоленной воде..."
2616,Y722243,7.319,6.620,1.970,0.322,1.970,0.491,1.922,96.73,"[яйцо, помидор, масло растительное, соль, пере...",Яйца разбивают и взбивают с солью и чёрным пер...


In [ ]:
weights = pd.read_csv("/Applications/python/nutristeppe/data/weight.csv", sep=';')
weights.rename(columns={"Порция": "serving_size_g", "code": "dish_category_code"}, inplace=True)
weights = weights[['dish_category_code', 'serving_size_g']]
weights

,dish_category_code,serving_size_g
0,GAR,150.0
1,GAR01,120.0
2,GAR0101,100.0
3,GAR0102,150.0
4,GAR0103,100.0
...,...,...
75,VEG02,250.0
76,BR,30.0
77,BR01,10.0
78,BR02,10.0


In [ ]:
df_new = pd.read_csv("/Applications/python/nutristeppe/data/type3_with_cat2.csv")
df_new

,bls_code,new_name,dish_category_code,type,type_5
0,B780200,Steiofebrot хлеб запеченный на каменных плитах,DGH0101,3,5
1,F201000,Абрикос,FR,3,1
2,F201100,Абрикос сырой,FR,3,5
3,F201101,Абрикос цельный,FR,3,5
4,F028700,Абрикосово-апельсиновый нектар,BEV04,3,4
...,...,...,...,...,...
5813,E108122,Яйцо пашот белок,EGG,2,5
5814,31105020,"Яйцо, жареное с маргарином",EGG,2,4
5815,17-743,Ячменная вода,BEV04,3,5
5816,51801010,Ячменный хлеб,DGH0102,3,2


In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "database": "balaman",
    "user": "timurchiks",
    "password": "",
    "port": "5432"
}

conn = psycopg2.connect(**DB_CONFIG)
query = "select * from dishes;"
dishes = pd.read_sql(query, conn)
dishes

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/3444354588.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dishes = pd.read_sql(query, conn)


,dish_id,bls_code,name,description,recipe_description,dish_category_id,dish_category_code,price,weight,kilocalories,...,carbohydrate,cfv,fvnl,index_health,type,type_5,new_name,cooking_technology,serving_method,quality_requirements
0,16564,F090400,Смешанные сушеные фрукты,None,None,43.0,DFR,0.0,25.0,287.0,...,0.00,70.0,5.0,3.50,3.0,5.0,Смешанные сушеные фрукты,None,None,None
1,16565,F090000,Смешанные фрукты,None,None,42.0,FR,0.0,125.0,85.0,...,0.00,30.0,5.0,4.00,3.0,5.0,Смешанные фрукты,None,None,None
2,16566,F090101,Смешанные фрукты сырые целые,None,None,42.0,FR,0.0,125.0,68.0,...,0.00,0.0,5.0,4.00,3.0,5.0,Смешанные свежие фрукты,None,None,None
3,19,X659012,"""Schupfudel"" (немецкая картофельная паста) (1)",None,None,5.0,GAR0201,0.0,200.0,100.0,...,0.00,0.0,100.0,3.75,2.0,2.0,Шупфнудель (немецкая картофельная паста),None,None,None
4,25,Y504322,"""Wildkrustel"" в панировке из дичи, обжаренная ...",None,None,7.0,MT01,0.0,300.0,141.0,...,0.00,0.0,0.0,2.75,2.0,2.0,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19320,9878,Y891312,Яблочные блины,None,None,16.0,DGH0201,0.0,100.0,194.9,...,25.46,0.0,30.0,2.25,2.0,1.0,Яблочные блины,"Муку смешивают с сахаром и солью, отдельно взб...","Подают горячими, можно посыпать сахаром или по...","Блины должны быть золотистого цвета, мягкие и ..."
19321,405,Y891433,Блинчики с творогом,None,None,38.0,CC,0.0,100.0,188.6,...,23.35,0.0,0.0,0.50,2.0,1.0,Блинчики с творогом,"В глубокой миске смешать муку, яйца, молоко и ...","Подают горячими, можно добавить свежие фрукты.","Блинчики должны быть золотистого цвета, мягкие..."
19322,899,Y892142,Сладкие картофельные клецки,None,None,3.0,GAR0103,0.0,100.0,178.5,...,37.54,26.3,26.3,4.00,2.0,1.0,Сладкие картофельные клецки,"Картофель отварить до полной готовности, воду ...","Подают горячими, можно посыпать сахаром или по...","Внешний вид: клецки одинаковой формы, без трещ..."
19323,3078,Y912130,Мясная котлета с картофельным салатом и горчицей,None,None,10.0,MT04,0.0,100.0,123.9,...,7.35,200.0,600.0,4.00,2.0,1.0,Мясная котлета с картофельным салатом и горчицей,Для приготовления котлеты говядину пропускаем ...,"Блюдо подают горячим, температура подачи 65-70°C.","Котлета должна быть золотистой, с хрустящей ко..."


In [ ]:
dishes.columns.to_list()

['dish_id',
 'bls_code',
 'name',
 'description',
 'recipe_description',
 'dish_category_id',
 'dish_category_code',
 'price',
 'weight',
 'kilocalories',
 'image_url',
 'has_relation_with_products',
 'health_factor',
 'created_at',
 'updated_at',
 'protein',
 'fat',
 'carbohydrate',
 'cfv',
 'fvnl',
 'index_health',
 'type',
 'type_5',
 'new_name',
 'cooking_technology',
 'serving_method',
 'quality_requirements']

In [ ]:
dishes = dishes[((dishes["type_5"] == 1) | (dishes["type_5"] == 2)) & ((dishes['type'] == 2) | (dishes['type'] == 3))]

In [ ]:
dishes['ingredients'] = None
dishes['techcard'] = None
dishes['fiber'] = None
dishes['sugar_mg'] = None
dishes['salt_total_mg'] = None
dishes['saturated_fat_mg'] = None
#sugar_mg, salt_total_mg, saturated_fat_mg

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/3605098529.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dishes['ingredients'] = None
/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/3605098529.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dishes['techcard'] = None
/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/3605098529.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

In [ ]:
dishes=dishes.drop(columns=['weight','image_url','name','dish_id','description','recipe_description','price','has_relation_with_products','health_factor','created_at','updated_at'])
dishes.rename(columns={"new_name": "name", "type_5": "availability_type", "cooking_technology": "steps"}, inplace=True)
df_new = df_new.rename(columns={"new_name": "name", "type_5": "availability_type"})

In [ ]:
dishes

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
3,X659012,5.0,GAR0201,100.0,0.00,0.00,0.00,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
4,Y504322,7.0,MT01,141.0,0.00,0.00,0.00,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
6,X880522,6.0,GAR0202,115.0,0.00,0.00,0.00,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
7,X693213,41.0,EGG,131.0,0.00,0.00,0.00,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
8,X468713,29.0,CMT02,96.0,0.00,0.00,0.00,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19320,Y891312,16.0,DGH0201,194.9,5.62,7.84,25.46,0.0,30.00,2.25,...,Яблочные блины,"Муку смешивают с сахаром и солью, отдельно взб...","Подают горячими, можно посыпать сахаром или по...","Блины должны быть золотистого цвета, мягкие и ...",None,None,None,None,None,None
19321,Y891433,38.0,CC,188.6,6.99,7.47,23.35,0.0,0.00,0.50,...,Блинчики с творогом,"В глубокой миске смешать муку, яйца, молоко и ...","Подают горячими, можно добавить свежие фрукты.","Блинчики должны быть золотистого цвета, мягкие...",None,None,None,None,None,None
19322,Y892142,3.0,GAR0103,178.5,5.28,0.81,37.54,26.3,26.30,4.00,...,Сладкие картофельные клецки,"Картофель отварить до полной готовности, воду ...","Подают горячими, можно посыпать сахаром или по...","Внешний вид: клецки одинаковой формы, без трещ...",None,None,None,None,None,None
19323,Y912130,10.0,MT04,123.9,10.48,5.85,7.35,200.0,600.00,4.00,...,Мясная котлета с картофельным салатом и горчицей,Для приготовления котлеты говядину пропускаем ...,"Блюдо подают горячим, температура подачи 65-70°C.","Котлета должна быть золотистой, с хрустящей ко...",None,None,None,None,None,None


In [ ]:
dishes['bls_code'] = dishes['bls_code'].astype(str).str.strip()
df_new['bls_code'] = df_new['bls_code'].astype(str).str.strip()

# Обновляем существующие строки
dishes_idx = dishes.set_index('bls_code')
df_new_idx = df_new.set_index('bls_code')
dishes_idx.update(df_new_idx)
dishes = dishes_idx.reset_index()

# Добавляем новые строки которых не было в dishes
new_codes = set(df_new['bls_code']) - set(dishes['bls_code'])
print(f"Новых кодов для добавления: {len(new_codes)}")  # проверяем сколько новых

new_rows = df_new[df_new['bls_code'].isin(new_codes)]
df_final = pd.concat([dishes, new_rows], ignore_index=True)

print(f"Строк до: {len(dishes)}, строк после: {len(df_final)}")
df_final

Новых кодов для добавления: 5291
Строк до: 6091, строк после: 11382


,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11377,51119010,NaN,DGH0102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яичный хлеб Хала,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11378,E108122,NaN,EGG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яйцо пашот белок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11379,31105020,NaN,EGG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Яйцо, жареное с маргарином",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11380,17-743,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Ячменная вода,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
json_df = json_df.set_index("bls_code")
df_final = df_final.set_index("bls_code")

df_final.update(json_df)

df_final = df_final.reset_index()
json_df = json_df.reset_index()

In [ ]:
# Находим индексы строк, которые нужно удалить
# Условие: type == 1 И type_5 входит в список [3, 4, 5]
to_drop = df_final[(df_final['type'] == 1) | (df_final['availability_type'].isin([3, 4, 5]))].index
print(len(to_drop))
# Удаляем эти строки
df_final = df_final.drop(to_drop)
df_final

5208


,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11341,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочно-морковный сок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11342,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный компот,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11343,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный компот с изюмом,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11346,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный сок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final = df_final.merge(weights, on="dish_category_code", how="left")
df_final

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,None,None,None,None,None,None,None,None,None,100.0
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,None,None,None,None,None,None,None,None,None,90.0
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,None,None,None,None,None,None,None,None,None,100.0
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,None,None,None,None,None,None,None,None,None,140.0
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,None,None,None,None,None,None,None,None,None,150.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0
6170,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0
6171,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0
6172,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0


In [ ]:
df_final[["protein", "fat", "carbohydrate"]] *= 1000

In [ ]:
df_final['kilocalories_portion'] = df_final['kilocalories'] * df_final['serving_size_g']/100

In [ ]:
# Создаем новую колонку 'Сумма_БЖУ'
df_final['calculated_kcal'] = ((df_final['protein'] + df_final['carbohydrate']) * 4 + (df_final['fat'] * 9))/1000
# Выбираем только интересующие нас данные
result_df = df_final[['bls_code', 'name','serving_size_g', 'dish_category_code', 'protein', 'fat', 'carbohydrate', 'calculated_kcal', 'kilocalories', 'kilocalories_portion']]
result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']
# ascending=False означает "по убыванию"
result_df = result_df[(result_df["protein"] > 0) & (result_df["fat"] > 0) & (result_df["carbohydrate"] > 0)]
result_df = result_df.sort_values(by='sub', ascending=False)

result_df

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/21352208.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']


,bls_code,name,serving_size_g,dish_category_code,protein,fat,carbohydrate,calculated_kcal,kilocalories,kilocalories_portion,sub
5499,D310200,Сырный торт из бисквитного теста,90.0,DES02,7568.0,11130.0,33680.0,265.162,265.3,238.77,0.138
3039,22002100,Котлета свиная в панировке,90.0,MT04,14670.0,17350.0,10760.0,257.870,258.0,232.20,0.130
3786,58109210,Пицца с яйцом,140.0,EGG,7560.0,4190.0,29830.0,187.270,187.4,262.36,0.130
3516,U952000,Свинина в панировке жареная без масла,90.0,MT02,14030.0,10310.0,24290.0,246.070,246.2,221.58,0.130
3608,53341750,"Пирог ""Шахматы""",100.0,DGH03,8610.0,21140.0,40170.0,385.380,385.5,385.50,0.120
...,...,...,...,...,...,...,...,...,...,...,...
1721,18-383,Жареный фазан,90.0,MT01,22320.0,10140.0,420.0,182.220,182.1,163.89,-0.120
5166,X035110,Хлебцы из цельнозерновой муки с маслом и полут...,150.0,SN04,15120.0,12740.0,35320.0,316.420,316.3,474.45,-0.120
5177,X286450,"Салат с рисом, яблоками и карри",150.0,SAL03,5340.0,2540.0,59000.0,280.220,280.1,420.15,-0.120
5302,19-612,Запеканка из свинины и яблок,150.0,CMT01,12830.0,13810.0,3004.0,187.626,187.5,281.25,-0.126


In [ ]:
df_final[df_final['name'] == 'Компот из голубой сливы'] #Y842612

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal
5878,Y842612,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN


In [ ]:
df_final[df_final['bls_code'] == 'Y842612']

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal
5878,Y842612,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN


In [ ]:
dishes[dishes['bls_code'] == 'Y842612']

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg


In [ ]:
df_new[df_new['bls_code'] == 'Y842612']

,bls_code,name,dish_category_code,type,availability_type
1502,Y842612,Компот из голубой сливы,BEV04,2,1


In [ ]:
json_df[json_df['bls_code'] == 'Y842612']

,bls_code,protein,fat,carbohydrate,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,kilocalories,ingredients,techcard


In [ ]:
print(df_new[df_new['name'].str.contains("Компот", na=False)]['bls_code'].values)
print(df_final[df_final['name'].str.contains("Компот", na=False)]['bls_code'].values)

['F951900' 'Y841443' 'Y842412' 'Y846312' 'F949900' 'Y842343' 'F930200'
 'Y852312' 'Y844912' 'Y839312' 'Y840543' 'Y840563' 'F401222' 'F945900'
 'F948900' 'F947900' 'Y842512' 'Y842012' 'Y841312' 'Y842612' 'Y840843'
 'Y840863' 'Y840853' 'Y840953' 'Y840643' 'Y840653' 'Y840663' 'F944200'
 'Y842253' 'Y841363' 'Y841953' 'Y841112' 'Y841212' 'Y840753' 'Y840743'
 'Y840443' 'F943232' 'Y836312' 'Y837312' 'Y843312' 'F952900' 'F955900'
 'F952932' 'Y848212' 'Y840143' 'Y840153' 'Y840163' 'F957200' 'Y851512'
 'Y853512' 'F940200' 'Y849912' 'Y841543' 'Y841553' 'Y842443' 'Y841263'
 'Y888312' 'Y840253' 'Y840243' 'Y840263' 'F926222' 'F942900' 'F926232'
 'Y841643' 'F954900' 'F953900' 'F954932' 'Y865312' 'F941232' 'F927900'
 'F927200' 'Y840363']
['Y841443' 'Y842343' 'F930200' 'Y840543' 'Y842512' 'Y842612' 'Y840843'
 'Y840643' 'Y841212' 'Y840753' 'Y840443' 'Y837312' 'Y843312' 'Y848212'
 'F940200' 'Y841263' 'Y840253' 'Y841643' 'F953900' 'F927200']


In [ ]:
df_final['bls_code'].isna().sum()

np.int64(0)

In [ ]:
df_final.to_csv("30March.csv", index=False)

In [ ]:
df_master = pd.read_csv("/Applications/python/nutristeppe/data/Master-BAZA-2_updated2222_from_json_fixed_20260330_172248.xlsx - Sheet1.csv")
print(df_master['Энергия (килокалории) ккал / 100г'].isna().sum())
df_master.rename(columns={'Энергия (килокалории) ккал / 100г': 'kilocalories'}, inplace=True)
df_master['kilocalories'] = pd.to_numeric(
    df_master['kilocalories'].astype(str).str.replace(',', '.').str.strip(), 
    errors='coerce'
)
df_master


/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/1996268915.py:1: DtypeWarning: Columns (0,1,2,8,10,15,17,19,20,21,22,28,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,137,138,139,145,146,147,148,149,150,151,152,153,154,156,157,158,159,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df_master = pd.read_csv("/Applications/python/nutristeppe/data/Master-BAZA-2_updated2222_from_json_fixed_20260330_172248.xlsx - Sheet1.csv")


5


,country_code,bls_code,name_german,name_en,name_old_ru,name,new_name,type,weight,5_category,...,Хлебные единицы,Поваренная соль всего,Средний размер порции г / порцию,Сезонность,"Селен, мкг","TFA, трансжирные кислоты, г","Криптоксантин, мкг","Лютеин, мкг","Ликопин, мкг",Unnamed: 165
0,NaN,M820000,Frischkäsezubereitung,Cream cheese preparation,Сливочный сыр приготовление,Сливочный сыр,Сливочный сыр,3.0,30,5.0,...,0.21,939,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,M820100,Frischkäsezubereitung < 10% Fett i. Tr.,Cream cheese preparation < 10 % fat in dry matter,Сливочный сыр приготовление <10% жирности,Сливочный сыр <10% жирности,Сливочный сыр не менее 10% жирности,3.0,30,5.0,...,0.26,102,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,M820300,Frischkäsezubereitung mind. 20% Fett i. Tr.,Cream cheese preparation min. 20 % fat in dry ...,Сливочный сыр приготовление не менее 20% жир...,Сливочный сыр не менее 20% жирности,Сливочный сыр не менее 20% жирности,3.0,30,5.0,...,0.3,99,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,M820700,Frischkäsezubereitung mind. 50% Fett i. Tr.,Cream cheese preparation min. 50 % fat in dry ...,Сливочный сыр приготовление не менее 50% жир...,Сливочный сыр не менее 50% жирности,Сливочный сыр не менее 50% жирности,3.0,30,5.0,...,0.21,705,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,M820800,Frischkäsezubereitung mind. 60% Fett i. Tr.,Cream cheese preparation min. 60 % fat in dry ...,Сливочный сыр приготовление не менее 60% жир...,Сливочный сыр не менее 60% жирности,Сливочный сыр не менее 60% жирности,3.0,30,5.0,...,0.21,933,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25905,KAZAKH,KZ58,NaN,NaN,Шубат,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25906,NaN,NaN,NaN,NaN,NaN,NaN,Протеиновый коктейль/Формула 1,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25907,NaN,NaN,NaN,NaN,NaN,NaN,Продукт белковый специализированный/Формула 3,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25908,NaN,NaN,NaN,NaN,NaN,NaN,БАД Фито Комплит,3.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_master = df_master[((df_master["5_category"] == 1) | (df_master["5_category"] == 2)) & (df_master['type'] == 3)]

In [ ]:
# Создаем новую колонку 'Сумма_БЖУ'
df_master['calculated_kcal'] = ((df_master['protein'] + df_master['carbohydrate']) * 4 + (df_master['fat'] * 9))/1000
# Выбираем только интересующие нас данные
result_df = df_master[['dish_category_code','bls_code', 'protein', 'fat', 'carbohydrate', 'calculated_kcal', 'kilocalories']]
result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']
# ascending=False означает "по убыванию"
result_df = result_df[(result_df["protein"] > 0) & (result_df["fat"] > 0) & (result_df["carbohydrate"] > 0)]
result_df = result_df.sort_values(by='sub', ascending=False)

result_df

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/1942046049.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master['calculated_kcal'] = ((df_master['protein'] + df_master['carbohydrate']) * 4 + (df_master['fat'] * 9))/1000
/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_57416/1942046049.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']


,bls_code,protein,fat,carbohydrate,calculated_kcal,kilocalories,sub
11105,X328563,4328.0,2799.0,2133.0,51.035,87.0,35.965
3731,F553400,11294.0,4706.0,32941.0,219.294,250.0,30.706
10638,F223400,3504.0,584.0,51398.0,224.864,253.0,28.136
5917,F609400,4654.0,1165.0,52159.0,237.737,262.0,24.263
8833,F225400,3651.0,462.0,56707.0,245.590,268.0,22.410
...,...,...,...,...,...,...,...
17912,43103000,20450.0,61210.0,11730.0,679.610,631.0,-48.610
17818,42102000,14320.0,67100.0,11740.0,708.140,659.0,-49.140
23217,17-051,600.0,4600.0,74400.0,341.400,281.0,-60.400
20878,75232000,31840.0,4010.0,52390.0,373.010,298.0,-75.010


In [ ]:
result_df.to_csv("brak.csv", index=False)

In [ ]:
print(df_master['kilocalories'].unique()[:20])

[356. 342. 349. 327. 348. 332. 355. 333. 350. 359. 377. 364. 368. 362.
 380. 337. 345. 320. 339. 347.]


In [ ]:
print(df_master['kilocalories'].isna().sum())   # количество NaN
print((df_master['kilocalories'] == 0).sum()) 

1
1


In [ ]:
df_master.describe()

,type,5_category,product_category_id,"Наличие в Генерации и/или Дневнике (1-Г+Д, 0-Д)",Заметки (5-заменить наименование),"Категория доступности (1-бытовая , 2-повседневная, 3-экзотика, 4-премиум)",Активно (1) или неактивно (0),kilocalories,Ингредиент (1) или блюдо (2),water,...,Докозадиеновая кислота мг / 100г,Докозатриеновая кислота мг / 100 г,Докозатетраеновая кислота мг / 100 г,Соотношение полиненасыщенных и насыщенных жиров (P / S),"Селен, мкг","TFA, трансжирные кислоты, г","Криптоксантин, мкг","Лютеин, мкг","Ликопин, мкг",calculated_kcal
count,1369.0,1369.000000,1365.000000,468.000000,0.0,468.000000,983.000000,1368.000000,987.000000,1368.000000,...,983.000000,983.000000,983.000000,983.000000,321.000000,107.000000,266.000000,30.000000,249.000000,1368.000000
mean,3.0,1.384953,28.832967,0.653846,NaN,1.476496,0.932859,204.237573,2.116515,55302.194444,...,0.046796,0.155646,0.756867,3897.018047,20285.082555,190.186916,17.003759,82.766667,654.570281,204.397847
std,0.0,0.486762,11.757568,0.476252,NaN,1.205574,0.250394,150.591858,0.638203,29668.183421,...,0.696110,1.798728,7.310602,12834.402752,22930.074694,420.365926,95.496577,108.703546,2454.821585,153.141300
min,3.0,1.000000,1.000000,0.000000,NaN,0.000000,0.000000,0.000000,1.000000,200.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000
25%,3.0,1.000000,19.000000,0.000000,NaN,0.000000,1.000000,74.000000,2.000000,35714.250000,...,0.000000,0.000000,0.000000,0.060000,2.000000,0.000000,0.000000,11.250000,0.000000,72.450000
50%,3.0,1.000000,26.000000,1.000000,NaN,2.000000,1.000000,175.000000,2.000000,62220.500000,...,0.000000,0.000000,0.000000,0.770000,39.000000,0.000000,0.000000,28.000000,0.000000,174.785000
75%,3.0,2.000000,40.000000,1.000000,NaN,2.000000,1.000000,329.000000,3.000000,81110.750000,...,0.000000,0.000000,0.000000,2.820000,46143.000000,110.000000,1.000000,132.750000,0.000000,329.615750
max,3.0,2.000000,50.000000,1.000000,NaN,4.000000,1.000000,729.000000,3.000000,99980.000000,...,18.000000,34.000000,112.000000,46359.000000,46295.000000,1810.000000,1395.000000,379.000000,19167.000000,740.183000


In [ ]:
df_master['kilocalories'].isna().sum()

np.int64(1)

In [ ]:
df_final = df_final.merge(df_master[['bls_code', 'name_en']], on="bls_code", how="left")
df_final

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal,name_en
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,None,None,None,None,None,None,100.0,100.0,0.0,NaN
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,None,None,None,None,None,None,90.0,126.9,0.0,NaN
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,None,None,None,None,None,None,100.0,115.0,0.0,NaN
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,None,None,None,None,None,None,140.0,183.4,0.0,NaN
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,None,None,None,None,None,None,150.0,144.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Apple/carrot fruit juice
6170,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,Apple compote (5)
6171,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,Apple compote with sultanas (4)
6172,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Apple juice


In [ ]:
df_final['name']

0                Шупфнудель (немецкая картофельная паста)
1       «Вильдкрустельн» из дичи в панировке, обжаренн...
2                                           Рис с горохом
3       Хоппель-Поппель (омлет с жареным картофелем и ...
4                 Гайсбургер Марш (картофель с говядиной)
                              ...                        
6169                                Яблочно-морковный сок
6170                                      Яблочный компот
6171                             Яблочный компот с изюмом
6172                                         Яблочный сок
6173                              Яблочный сок без сахара
Name: name, Length: 6174, dtype: object

In [ ]:
df_final.columns

Index(['bls_code', 'dish_category_id', 'dish_category_code', 'kilocalories',
       'protein', 'fat', 'carbohydrate', 'cfv', 'fvnl', 'index_health', 'type',
       'availability_type', 'name', 'steps', 'serving_method',
       'quality_requirements', 'ingredients', 'techcard', 'fiber', 'sugar_mg',
       'salt_total_mg', 'saturated_fat_mg', 'serving_size_g',
       'kilocalories_portion', 'calculated_kcal', 'name_en'],
      dtype='object')

In [ ]:
df_final['dish_category_id'].isna().sum()

np.int64(487)

In [ ]:
# Мержим нужные колонки из df_master
cols_from_master = ['bls_code', 'protein', 'fat', 'carbohydrate', 'kilocalories']
df_merged = df_final.merge(df_master[cols_from_master], on='bls_code', how='left', suffixes=('', '_master'))

# Считаем ккал из БЖУ мастера
df_merged['calculated_kcal'] = ((df_merged['protein_master'] + df_merged['carbohydrate_master']) * 4 + (df_merged['fat_master'] * 9)) / 1000

# Считаем разницу
df_merged['sub'] = df_merged['kilocalories_master'] - df_merged['calculated_kcal']

# Только строки где БЖУ больше нуля
df_merged = df_merged[(df_merged['protein_master'] > 0) & (df_merged['fat_master'] > 0) & (df_merged['carbohydrate_master'] > 0)]

# Разделяем на брак и нормальные
df_brak = df_merged[df_merged['sub'].abs() > 5]
df_good = df_merged[df_merged['sub'].abs() <= 5]

# Обновляем df_final только хорошими данными
df_final = df_final.set_index('bls_code')
df_good_idx = df_good.set_index('bls_code')
df_final.update(df_good_idx[['protein_master', 'fat_master', 'carbohydrate_master', 'kilocalories_master']])
df_final = df_final.reset_index()

print(f"Хороших записей: {len(df_good)}")
print(f"Брак: {len(df_brak)}")

Хороших записей: 330
Брак: 78


In [ ]:
print(df_master[['protein', 'fat', 'carbohydrate', 'kilocalories']].head(3))
print(df_final[['protein', 'fat', 'carbohydrate', 'kilocalories']].head(3))

    protein     fat  carbohydrate  kilocalories
10  12982.0  2283.0       69759.0         356.0
12  10533.0  1935.0       69201.0         342.0
13  12262.0  1828.0       69667.0         349.0
   protein  fat  carbohydrate  kilocalories
0      0.0  0.0           0.0         100.0
1      0.0  0.0           0.0         141.0
2      0.0  0.0           0.0         115.0


In [ ]:
df_final.to_csv("31March.csv", index=False)

In [ ]:
cols = ['protein', 'fat', 'carbohydrate', 'kilocalories']
df_nulls = df_final[df_final[cols].isna().any(axis=1)]
df_nulls

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal,name_en
5712,F201000,NaN,FR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,Apricot
5713,Y841453,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Apricot compote (5)
5714,X365543,NaN,SAU02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN
5715,F501000,NaN,FR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,Pineapple
5716,F603000,NaN,FR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NaN,NaN,Orange
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Apple/carrot fruit juice
6170,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,Apple compote (5)
6171,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,Apple compote with sultanas (4)
6172,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN,Apple juice


In [ ]:
df_nulls.to_csv("weird.csv", index=False)